In [6]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings

warnings.filterwarnings('ignore')
print("All libraries imported.")

All libraries imported.


In [7]:
data = load_breast_cancer()
X = data.data
y = data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split and scaled.")

Data split and scaled.


In [9]:
scaler_filename = "scaler.joblib"
joblib.dump(scaler, scaler_filename)
print(f"Scaler saved to {scaler_filename}")

Scaler saved to scaler.joblib


In [10]:
mlflow.set_experiment("cancer-project")

2025/11/16 19:52:25 INFO mlflow.tracking.fluent: Experiment with name 'cancer-project' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/alfarouq/Library/CloudStorage/OneDrive-Universite%CC%81MohammedVIPolytechnique/S9/cloud%20computing/lab2/Lab%20Folder%20MLOps/mlruns/711509467056098314', creation_time=1763319145520, experiment_id='711509467056098314', last_update_time=1763319145520, lifecycle_stage='active', name='cancer-project', tags={}>

### Logistic Regression

In [11]:
with mlflow.start_run(run_name="LogisticRegression"):
    C = 1.0
    solver = 'liblinear'
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("C", C)
    mlflow.log_param("solver", solver)
    
    model_lr = LogisticRegression(C=C, solver=solver, random_state=42)
    model_lr.fit(X_train_scaled, y_train)
    
    y_pred = model_lr.predict(X_test_scaled)
    y_proba = model_lr.predict_proba(X_test_scaled)[:,-1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)
    
    print(f"Logistic Regression: Accuracy = {acc:.4f}, AUC = {auc:.4f}")
    
    mlflow.sklearn.log_model(model_lr, "model")
    mlflow.log_artifact(scaler_filename)
    
    print("Logged Logistic Regression model and scaler.joblib artifact.")

2025/11/16 19:52:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logistic Regression: Accuracy = 0.9825, AUC = 0.9957


2025/11/16 19:52:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/16 19:52:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged Logistic Regression model and scaler.joblib artifact.


### Random Forest

In [ ]:
with mlflow.start_run(run_name="RandomForest"):
    
    n_estimators = 100
    max_depth = 5
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    model_rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    
    model_rf.fit(X_train_scaled, y_train)
    
    y_pred = model_rf.predict(X_test_scaled)
    y_proba = model_rf.predict_proba(X_test_scaled)[:,-1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)
    
    print(f"Random Forest: Accuracy = {acc:.4f}, AUC = {auc:.4f}")

    mlflow.sklearn.log_model(model_rf, "model")
    
    print("Logged Random Forest model.")

2025/11/16 19:53:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest: Accuracy = 0.9561, AUC = 0.9934


2025/11/16 19:53:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/16 19:53:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged Random Forest model.
